# 15.5 — Equazione delle onde

**Prerequisiti:** 15.4 Equazione del calore, 15.2 Trasformata di Fourier (rigorosa), 8.4 Equazioni a coefficienti costanti.


## Perché ci interessa

Nel notebook 15.4 l'equazione del calore ha mostrato un comportamento "dissipativo": le singolarità si smussano istantaneamente. L'**equazione delle onde** $\partial_t^2u=\partial_x^2u$ descrive un fenomeno fisicamente opposto — la propagazione di un segnale (il suono, la luce, una corda vibrante) **senza perdita di informazione**: le singolarità del dato iniziale **viaggiano** invece di dissolversi. Questa equazione è il prototipo delle equazioni **iperboliche** (notebook 15.10), e la sua analisi tramite trasformata di Fourier porta a una scoperta notevole: una formula esplicita e chiusa (D'Alembert) che rende superfluo, in questo caso particolare, il macchinario della trasformata — ma che la trasformata stessa aiuta a **scoprire** e comprendere in profondità.

## Teoria, passo dopo passo

### Il problema di Cauchy per l'equazione delle onde

Cerchiamo $u(x,t)$ soluzione di $\partial_t^2u=\partial_x^2u$, con **due** dati iniziali (essendo l'equazione del secondo ordine in $t$, servono sia posizione sia velocità iniziale, come per un'equazione differenziale ordinaria del secondo ordine, notebook 8.4): $u(x,0)=f(x)$, $\partial_tu(x,0)=g(x)$.

### Risoluzione tramite trasformata di Fourier

Trasformando in $x$: $\partial_t^2\hat u(\xi,t)=-4\pi^2\xi^2\hat u(\xi,t)$ (stessa proprietà derivazione↔moltiplicazione del notebook 15.2, applicata a $\partial_x^2$). Per $\xi$ fissato, questa è un'**equazione differenziale ordinaria lineare del secondo ordine a coefficienti costanti** (notebook 8.4): $\hat u''+4\pi^2\xi^2\hat u=0$, con equazione caratteristica $r^2+4\pi^2\xi^2=0$, cioè $r=\pm2\pi i\xi$ (radici immaginarie pure, notebook 8.4, caso oscillatorio). La soluzione generale è $\hat u(\xi,t)=A(\xi)\cos(2\pi\xi t)+B(\xi)\sin(2\pi\xi t)$.

Imponendo le condizioni iniziali: $\hat u(\xi,0)=A(\xi)=\hat f(\xi)$; $\partial_t\hat u(\xi,0)=2\pi\xi B(\xi)=\hat g(\xi)$, quindi $B(\xi)=\hat g(\xi)/(2\pi\xi)$ (per $\xi\ne0$). Dunque $\hat u(\xi,t)=\hat f(\xi)\cos(2\pi\xi t)+\dfrac{\hat g(\xi)}{2\pi\xi}\sin(2\pi\xi t)$.

### La formula di D'Alembert

Usando $\cos(2\pi\xi t)=\dfrac{e^{2\pi i\xi t}+e^{-2\pi i\xi t}}2$ (Eulero, notebook 9.3) e riconoscendo che moltiplicare per $e^{2\pi i\xi t}$ in frequenza corrisponde a **traslare** nello spazio fisico (proprietà della trasformata di Fourier, dimostrabile con un cambio di variabile diretto nell'integrale che la definisce): $\widehat{f(\cdot+t)}(\xi)=e^{2\pi i\xi t}\hat f(\xi)$. Applicando questo al primo termine: l'antitrasformata di $\hat f(\xi)\cos(2\pi\xi t)$ è $\dfrac{f(x+t)+f(x-t)}2$. Un calcolo analogo (che coinvolge un'integrazione, per via del fattore $1/\xi$) dà, per il secondo termine, $\dfrac12\int_{x-t}^{x+t}g(y)\,dy$. Sommando:
$$u(x,t)=\dfrac{f(x-t)+f(x+t)}2+\dfrac12\int_{x-t}^{x+t}g(y)\,dy.$$
Questa è la celebre **formula di D'Alembert**, che risolve esplicitamente l'equazione delle onde in una dimensione spaziale.

### Propagazione a velocità finita: il cono di luce

**Osservazione cruciale.** $u(x,t)$ dipende **solo** dai valori di $f,g$ sull'intervallo $[x-t,x+t]$: un'informazione localizzata in un punto $x_0$ al tempo $0$ (per esempio, una singolarità di $f$ in $x_0$) può influenzare $u(x,t)$ solo se $x_0\in[x-t,x+t]$, cioè $|x-x_0|\le t$. Questo è il fenomeno della **velocità finita di propagazione** (qui, velocità $1$, coerente con i coefficienti dell'equazione): l'informazione si propaga lungo le **rette caratteristiche** $x\pm t=\text{costante}$, mai più velocemente. Questo è **radicalmente diverso** dall'equazione del calore (notebook 15.4), dove ogni punto del dato iniziale influenza istantaneamente **ogni** punto della soluzione (essendo $K_t(x)>0$ ovunque, per quanto piccolo).

### Le singolarità persistono: nessun effetto regolarizzante

**Conseguenza della formula di D'Alembert.** Se $f$ ha una discontinuità (per esempio, nella derivata) in $x_0$, la formula $u(x,t)=[f(x-t)+f(x+t)]/2+\cdots$ mostra che questa discontinuità si **propaga** lungo le rette $x-x_0=\pm t$, senza mai attenuarsi: al contrario del calore (notebook 15.4), l'equazione delle onde **non ha effetto regolarizzante**. Questo è il tratto distintivo delle equazioni iperboliche rispetto a quelle paraboliche, un contrasto che approfondiremo sistematicamente nei notebook 15.9-15.10.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return np.exp(-x**2 * 5)  # dato iniziale liscio, concentrato vicino a 0

def g(x):
    return np.zeros_like(x)  # velocita' iniziale nulla

def u_dalembert(x, t, f, g, dx=0.001):
    parte_f = (f(x - t) + f(x + t)) / 2
    y_integr = np.arange(x - t, x + t, dx) if t > 0 else np.array([x])
    parte_g = np.trapz(g(y_integr), y_integr) / 2 if t > 0 else 0
    return parte_f + parte_g

x = np.linspace(-5, 5, 300)
fig, ax = plt.subplots(figsize=(8,5))
for t in [0, 1, 2, 3]:
    u_t = np.array([u_dalembert(xi, t, f, g) for xi in x])
    ax.plot(x, u_t, label=f"t={t}")
ax.legend()
ax.set_title("Equazione delle onde: il pacchetto si divide e viaggia, SENZA smussarsi")
plt.tight_layout()
plt.show()

print("A differenza del calore, il picco iniziale si divide in due pacchetti che viaggiano")
print("a velocita' 1 in direzioni opposte, mantenendo (in questo caso a g=0) la propria forma.")

## Esercizio svolto e commentato

**Problema:** usa la formula di D'Alembert per risolvere l'equazione delle onde con $f(x)=0$ (posizione iniziale nulla) e $g(x)=\mathbb 1_{[-1,1]}(x)$ (velocità iniziale a gradino), e determina esplicitamente $u(0,t)$ per $t>2$.

**Ragionamento:**

**Passo 1 — applichiamo direttamente la formula.** Con $f=0$: $u(x,t)=\dfrac12\int_{x-t}^{x+t}g(y)\,dy$.

**Passo 2 — valutiamo in $x=0$.** $u(0,t)=\dfrac12\int_{-t}^tg(y)\,dy=\dfrac12\int_{-t}^t\mathbb 1_{[-1,1]}(y)\,dy$.

**Passo 3 — calcoliamo l'integrale per $t>2$ (quindi $t>1$, l'intervallo $[-t,t]$ contiene interamente $[-1,1]$).** $\int_{-t}^t\mathbb 1_{[-1,1]}(y)\,dy=\int_{-1}^1 1\,dy=2$ (l'integranda vale $1$ solo su $[-1,1]\subseteq[-t,t]$, e $0$ altrove in $[-t,t]$).

**Passo 4 — concludiamo.** $u(0,t)=\dfrac12\cdot2=1$ per ogni $t>2$ (in realtà, basta $t\ge1$ per avere $[-1,1]\subseteq[-t,t]$; per $t>2$ è certamente vero, ma il risultato $u(0,t)=1$ vale già per $t\ge1$).

**Conclusione:** $u(0,t)=1$ per $t\ge1$: una volta che il "fronte d'onda" della perturbazione iniziale $g$ ha avuto tempo di raggiungere e superare il punto $x=0$ (che richiede tempo $t=1$, essendo il bordo di $[-1,1]$ a distanza $1$ da $0$, coerente con la velocità di propagazione $1$ discussa nella teoria), il valore si stabilizza. **Confronto con il calore:** questo comportamento "a gradino nel tempo" (nullo poi costante, con una transizione netta a $t=1$) è impossibile per l'equazione del calore, dove ogni effetto è istantaneo (anche se piccolo) fin da $t=0^+$, per l'effetto regolarizzante e la positività ovunque del nucleo $K_t$ (notebook 15.4).

In [ ]:
import numpy as np

def g(y):
    return np.where(np.abs(y) <= 1, 1.0, 0.0)

def u_0_t(t, dy=0.0001):
    y = np.arange(-t, t, dy)
    if len(y) == 0:
        return 0.0
    return 0.5 * np.trapz(g(y), y)

for t in [0.5, 0.9, 1.0, 1.5, 2, 3]:
    val = u_0_t(t)
    print(f"t={t}: u(0,t) = {val:.4f}")

print("\nVerifica: per t<1 il valore cresce (il fronte d'onda sta ancora 'entrando'),")
print("per t>=1 si stabilizza esattamente a 1, come predetto dal calcolo esatto")

## Esercizi da risolvere

1. Verifica che $u(x,t)=\sin(x-t)$ soddisfa $\partial_t^2u=\partial_x^2u$ (un'onda che viaggia a velocità $1$ verso destra, senza cambiare forma).
2. Usando D'Alembert con $f(x)=\sin(x)$, $g=0$, verifica che si ottiene $u(x,t)=\sin(x)\cos(t)$ (suggerimento: usa la formula di prostaferesi per $\sin(x-t)+\sin(x+t)$).
3. Dimostra, usando la formula di D'Alembert, il principio di **conservazione dell'energia**: se $f,g$ sono sufficientemente regolari e a supporto compatto, $E(t)=\dfrac12\int_{\mathbb R}\big[(\partial_tu)^2+(\partial_xu)^2\big]\,dx$ è **costante** nel tempo (suggerimento: deriva $E(t)$ rispetto a $t$, usa l'equazione delle onde per sostituire $\partial_t^2u$, e integra per parti).
4. *(più impegnativo)* Usa la formula esplicita $\hat u(\xi,t)=\hat f(\xi)\cos(2\pi\xi t)+\dfrac{\hat g(\xi)}{2\pi\xi}\sin(2\pi\xi t)$ per dimostrare che $\|u(\cdot,t)\|_{H^1}$ e $\|\partial_tu(\cdot,t)\|_{L^2}$ restano **limitati** uniformemente nel tempo (a differenza, per esempio, di equazioni con termini di crescita esponenziale) — un'altra manifestazione, nel linguaggio degli spazi di Sobolev (notebook 15.3), della conservazione dell'energia dell'esercizio 3.

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $\partial_tu=-\cos(x-t)$, $\partial_t^2u=-\sin(x-t)$ (derivando due volte rispetto a $t$, con la regola della catena, il segno si inverte due volte tornando a $-\sin$). $\partial_xu=\cos(x-t)$, $\partial_x^2u=-\sin(x-t)$. Dunque $\partial_t^2u=-\sin(x-t)=\partial_x^2u$: l'equazione è soddisfatta.

**2.** $u(x,t)=\dfrac{\sin(x-t)+\sin(x+t)}2$ (formula di D'Alembert con $g=0$). Per la formula di prostaferesi $\sin A+\sin B=2\sin\big(\frac{A+B}2\big)\cos\big(\frac{A-B}2\big)$ (Modulo 2), con $A=x-t$, $B=x+t$: $\sin(x-t)+\sin(x+t)=2\sin(x)\cos(t)$ (essendo $(A+B)/2=x$, $(A-B)/2=-t$, e $\cos(-t)=\cos(t)$). Dunque $u(x,t)=\dfrac{2\sin(x)\cos(t)}2=\sin(x)\cos(t)$.

**3.** $E'(t)=\int\big[2\partial_tu\cdot\partial_t^2u+2\partial_xu\cdot\partial_t\partial_xu\big]\,dx/2=\int\big[\partial_tu\cdot\partial_t^2u+\partial_xu\cdot\partial_x\partial_tu\big]\,dx$ (scambiando $\partial_t\partial_x=\partial_x\partial_t$, lecito per funzioni sufficientemente regolari, teorema di Schwarz, Modulo 7). Usando $\partial_t^2u=\partial_x^2u$ (l'equazione delle onde) nel primo termine: $E'(t)=\int\big[\partial_tu\cdot\partial_x^2u+\partial_xu\cdot\partial_x\partial_tu\big]\,dx$. Integrando per parti il primo termine (Modulo 5, con i termini di bordo nulli per supporto compatto): $\int\partial_tu\cdot\partial_x^2u\,dx=-\int\partial_x\partial_tu\cdot\partial_xu\,dx$. Sostituendo: $E'(t)=-\int\partial_x\partial_tu\cdot\partial_xu\,dx+\int\partial_xu\cdot\partial_x\partial_tu\,dx=0$ (i due termini sono opposti e si cancellano). Dunque $E(t)$ è costante.

**4.** $\|u(\cdot,t)\|_{H^1}^2=\int(1+\xi^2)|\hat u(\xi,t)|^2\,d\xi$, con $|\hat u(\xi,t)|^2=|\hat f(\xi)\cos(2\pi\xi t)+\hat g(\xi)\sin(2\pi\xi t)/(2\pi\xi)|^2\le2|\hat f(\xi)|^2\cos^2(2\pi\xi t)+2|\hat g(\xi)|^2\sin^2(2\pi\xi t)/(4\pi^2\xi^2)\le2|\hat f(\xi)|^2+2|\hat g(\xi)|^2/(4\pi^2\xi^2)$ (disuguaglianza $(a+b)^2\le2a^2+2b^2$, notebook 12.7, poi $|\cos|,|\sin|\le1$). Dunque $\|u(\cdot,t)\|_{H^1}^2\le2\|f\|_{H^1}^2+2\int(1+\xi^2)|\hat g(\xi)|^2/(4\pi^2\xi^2)\,d\xi$, una quantità **finita e indipendente da $t$** (assumendo $g$ sufficientemente regolare da rendere finito l'ultimo integrale): la norma $H^1$ di $u(\cdot,t)$ resta limitata uniformemente nel tempo, coerentemente con la conservazione dell'energia dell'esercizio 3 (che è, in sostanza, una versione più precisa — con costante esatta, non solo limitatezza — di questa stima).

</details>

## Riepilogo finale

- L'**equazione delle onde** $\partial_t^2u=\partial_x^2u$, trasformata in frequenza, diventa una famiglia di ODE lineari del secondo ordine **oscillatorie** (a differenza dell'ODE dissipativa del calore, notebook 15.4).
- La **formula di D'Alembert** $u(x,t)=\dfrac{f(x-t)+f(x+t)}2+\dfrac12\int_{x-t}^{x+t}g$ risolve esplicitamente il problema in una dimensione.
- **Velocità finita di propagazione**: l'informazione viaggia lungo le rette caratteristiche $x\pm t=\text{cost}$, senza influenzare istantaneamente tutto lo spazio — l'opposto del comportamento del calore.
- **Nessun effetto regolarizzante**: le singolarità del dato iniziale persistono, propagandosi senza attenuarsi; la conservazione dell'energia garantisce comunque un controllo uniforme nel tempo della norma della soluzione.

## Collegamenti

Il prossimo notebook è **15.6 — Equazione di Laplace**: passiamo da equazioni di evoluzione (dipendenti dal tempo) a un'equazione **stazionaria**, che descrive stati di equilibrio, introducendo le funzioni armoniche e le loro notevoli proprietà di media.